# Lab 2 (completed) — Flow Matching and Score Matching

Gaussian & linear conditional probability paths, and the flow- / score-matching training objectives on 2-D toy distributions.

In [1]:
import torch
from flow_matching_labs.core import EulerSimulator, build_ts
from flow_matching_labs.distributions import GaussianMixture
from flow_matching_labs.paths import GaussianConditionalProbabilityPath, LinearAlpha, SquareRootBeta
from flow_matching_labs.models import MLPVectorField, ConditionalFlowMatchingTrainer, LearnedVectorFieldODE
torch.manual_seed(0)

## Q3.1 Flow matching with a Gaussian conditional path

Train $u_t^\theta$ to match the conditional vector field, then integrate the learned ODE from noise to data.

In [2]:
target = GaussianMixture.symmetric_2D(nmodes=5, std=1.0, scale=10.0)
path = GaussianConditionalProbabilityPath(target, LinearAlpha(), SquareRootBeta())
model = MLPVectorField(dim=2, hiddens=[64, 64, 64, 64])
losses = ConditionalFlowMatchingTrainer(path, model).train(
    num_epochs=2000, device='cpu', lr=1e-3, batch_size=1000, progress=False)
print("final FM loss:", round(losses[-1], 3))

sim = EulerSimulator(LearnedVectorFieldODE(model))
gen = sim.simulate(path.p_simple.sample(5000), build_ts(500, 5000, 'cpu'))
print("generated mean/std:", gen.mean(0).tolist(), gen.std(0).tolist())

final FM loss: 21.658


generated mean/std: [-0.14317558705806732, -0.15832120180130005] [6.988314628601074, 7.334346771240234]


See `scripts/run_lab2.py` for the full experiments (score matching + Langevin SDE sampling, and the linear-path checkerboard) with saved density-match figures under `results/lab2/`.